# XGBoost–SARIMA model experiment
Production-oriented hybrid experiment with calendar/exogenous features, lag-52, separate holiday blend weights, chronological validation, and a portable artifact for inference.

In [ ]:
%pip install -q "xgboost>=3,<4" "statsmodels>=0.14,<1" "scikit-learn>=1.6,<2" "wandb>=0.19,<1" "cloudpickle>=3,<4"

In [ ]:
from pathlib import Path
import warnings
import cloudpickle
import numpy as np
import pandas as pd
import wandb
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.base import BaseEstimator, RegressorMixin, TransformerMixin
from sklearn.pipeline import Pipeline
warnings.filterwarnings('ignore')
DATA_DIR = Path('/content/drive/MyDrive/walmart_competition_data') if Path('/content').exists() else Path('../../data')
OUTPUT_DIR = Path('/content/drive/MyDrive/walmart_models') if Path('/content').exists() else Path('artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_WEEKS = 39
ORDERS = [
    ((1, 0, 1), (0, 1, 1, 52)),
    ((0, 1, 1), (0, 1, 1, 52)),
]

train_raw = pd.read_csv(DATA_DIR / 'train.csv', parse_dates=['Date'])
test_raw = pd.read_csv(DATA_DIR / 'test.csv', parse_dates=['Date'])
features = pd.read_csv(DATA_DIR / 'features.csv', parse_dates=['Date'])
stores = pd.read_csv(DATA_DIR / 'stores.csv')

def merge_tables(frame):
    return (
        frame.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left', validate='many_to_one')
        .merge(stores, on='Store', how='left', validate='many_to_one')
    )

train = merge_tables(train_raw)
test = merge_tables(test_raw)

In [ ]:
BASE=['Store','Dept','IsHoliday','Size','Temperature','Fuel_Price','CPI','Unemployment','MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
def make_x(df,history=None):
    z=df.copy(); z['Type']=z.Type.map({'A':0,'B':1,'C':2}); z['year']=z.Date.dt.year; z['month']=z.Date.dt.month
    z['week']=z.Date.dt.isocalendar().week.astype(int); z['week_sin']=np.sin(2*np.pi*z.week/52); z['week_cos']=np.cos(2*np.pi*z.week/52)
    if history is not None:
        lag=history[['Store','Dept','Date','Weekly_Sales']].copy(); lag.Date=lag.Date+pd.Timedelta(weeks=52); lag=lag.rename(columns={'Weekly_Sales':'lag_52'})
        z=z.merge(lag,on=['Store','Dept','Date'],how='left')
    else: z['lag_52']=np.nan
    cols=BASE+['Type','year','month','week','week_sin','week_cos','lag_52']
    return z[cols].astype(float).fillna(-999),cols
def wmae(y,p,h): return np.average(np.abs(np.asarray(y)-p),weights=np.where(np.asarray(h),5.,1.))
dates=np.sort(train.Date.unique()); cut=dates[-VALIDATION_WEEKS]; tr=train[train.Date<cut].copy(); va=train[train.Date>=cut].copy()
Xtr,cols=make_x(tr,tr); Xva,_=make_x(va,tr)
model=XGBRegressor(n_estimators=1600,max_depth=9,min_child_weight=8,learning_rate=.025,subsample=.85,colsample_bytree=.8,reg_lambda=4,objective='reg:absoluteerror',tree_method='hist',random_state=42)
model.fit(Xtr,tr.Weekly_Sales,sample_weight=np.where(tr.IsHoliday,5.,1.)); px=model.predict(Xva)

In [ ]:
def sarima_predictions(history,future,order,seasonal):
    out=np.full(len(future),np.nan)
    for key,g in future.groupby(['Store','Dept'],sort=False):
        y=history[(history.Store==key[0])&(history.Dept==key[1])].sort_values('Date').Weekly_Sales
        if len(y)<80: continue
        try: out[future.index.get_indexer(g.index)]=SARIMAX(y,order=order,seasonal_order=seasonal,enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=60).forecast(len(g))
        except Exception: pass
    return out
best=None
for order,seasonal in ORDERS:
    ps=sarima_predictions(tr,va,order,seasonal); ps=np.where(np.isfinite(ps),ps,px)
    for normal_w in np.arange(.5,.91,.1):
      for holiday_w in np.arange(.65,.96,.1):
        weights=np.where(va.IsHoliday,holiday_w,normal_w); blend=weights*px+(1-weights)*ps; score=wmae(va.Weekly_Sales,blend,va.IsHoliday)
        if best is None or score<best['score']: best={'score':score,'order':order,'seasonal_order':seasonal,'normal_xgb_weight':float(normal_w),'holiday_xgb_weight':float(holiday_w)}
print(best)

In [ ]:
# Refit XGBoost on all labeled rows.
Xall,cols=make_x(train,train); final_model=model.set_params(n_estimators=max(300,int(model.n_estimators*len(train)/len(tr))))
final_model.fit(Xall,train.Weekly_Sales,sample_weight=np.where(train.IsHoliday,5.,1.))
print({'selected_estimators': final_model.n_estimators, **best})

## Register the best hybrid pipeline
Run this only after validating the experiment. It packages preprocessing, the fitted XGBoost model, SARIMA configuration, history, and learned blend weights into one raw-input pipeline and links it to W&B Model Registry with the `champion` alias.

In [ ]:
REGISTER_MODEL = True
WANDB_PROJECT = 'Walmart-Recruiting---Store-Sales-Forecasting'
REGISTRY_TARGET = 'wandb-registry-model/Walmart_XGBoost_SARIMA_Pipeline'

class RawWalmartHybridTransformer(BaseEstimator, TransformerMixin):
    """Convert raw Walmart rows into model features while preserving SARIMA keys."""

    def __init__(self, external_features, stores):
        self.external_features = external_features
        self.stores = stores

    def _merge(self, raw):
        frame = raw.copy()
        frame['__input_order'] = np.arange(len(frame))
        frame['Date'] = pd.to_datetime(frame['Date'])
        frame = frame.merge(self.external_features, on=['Store','Date','IsHoliday'], how='left').merge(self.stores, on='Store', how='left')
        return frame.sort_values('__input_order').drop(columns='__input_order').reset_index(drop=True)

    def fit(self, X, y=None):
        if y is None:
            raise ValueError('Weekly_Sales must be supplied as y.')
        merged = self._merge(X)
        self.history_ = merged[['Store','Dept','Date']].copy()
        self.history_['Weekly_Sales'] = np.asarray(y)
        self.fill_values_ = {column: float(merged[column].median()) for column in BASE if column in merged.columns and column != 'IsHoliday'}
        self.feature_columns_ = list(cols)
        return self

    def transform(self, X):
        frame = self._merge(X)
        frame['Type'] = frame['Type'].map({'A':0,'B':1,'C':2})
        frame['year'] = frame.Date.dt.year
        frame['month'] = frame.Date.dt.month
        frame['week'] = frame.Date.dt.isocalendar().week.astype(int)
        frame['week_sin'] = np.sin(2*np.pi*frame.week/52)
        frame['week_cos'] = np.cos(2*np.pi*frame.week/52)
        lag = self.history_.copy()
        lag['Date'] = pd.to_datetime(lag['Date']) + pd.Timedelta(weeks=52)
        lag = lag.rename(columns={'Weekly_Sales':'lag_52'})
        frame = frame.merge(lag, on=['Store','Dept','Date'], how='left', validate='many_to_one')
        for column, value in self.fill_values_.items():
            if column in frame.columns:
                frame[column] = frame[column].fillna(value)
        output = frame[self.feature_columns_].astype(float).fillna(-999)
        output['__Store'] = frame['Store'].to_numpy()
        output['__Dept'] = frame['Dept'].to_numpy()
        output['__Date'] = frame['Date'].to_numpy()
        output['__IsHoliday'] = frame['IsHoliday'].to_numpy(dtype=bool)
        return output

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_columns_, dtype=object)


class XGBoostSARIMARegressor(BaseEstimator, RegressorMixin):
    """Blend a fitted global XGBoost model with per-series SARIMA forecasts."""

    def __init__(self, xgb_model, history, order, seasonal_order, normal_xgb_weight, holiday_xgb_weight, min_points=80):
        self.xgb_model = xgb_model
        self.history = history
        self.order = order
        self.seasonal_order = seasonal_order
        self.normal_xgb_weight = normal_xgb_weight
        self.holiday_xgb_weight = holiday_xgb_weight
        self.min_points = min_points

    def fit(self, X, y=None):
        self.feature_columns_ = [column for column in X.columns if not column.startswith('__')]
        return self

    def predict(self, X):
        xgb_pred = self.xgb_model.predict(X[self.feature_columns_])
        sarima_pred = np.full(len(X), np.nan)
        groups = X.groupby(['__Store','__Dept'], sort=False).groups
        for key, index in groups.items():
            series = self.history[(self.history.Store==key[0]) & (self.history.Dept==key[1])].sort_values('Date').Weekly_Sales
            if len(series) < self.min_points:
                continue
            try:
                fitted = SARIMAX(series, order=self.order, seasonal_order=self.seasonal_order, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False, maxiter=60)
                sarima_pred[X.index.get_indexer(index)] = fitted.forecast(len(index))
            except Exception:
                continue
        sarima_pred = np.where(np.isfinite(sarima_pred), sarima_pred, xgb_pred)
        weights = np.where(X['__IsHoliday'], self.holiday_xgb_weight, self.normal_xgb_weight)
        return weights*xgb_pred + (1-weights)*sarima_pred

if REGISTER_MODEL:
    raw_transformer = RawWalmartHybridTransformer(features, stores).fit(train_raw, train_raw['Weekly_Sales'])
    hybrid_estimator = XGBoostSARIMARegressor(final_model, train[['Store','Dept','Date','Weekly_Sales']], best['order'], best['seasonal_order'], best['normal_xgb_weight'], best['holiday_xgb_weight']).fit(raw_transformer.transform(train_raw.head(1)))
    pipeline = Pipeline([('feature_engineering', raw_transformer), ('hybrid_model', hybrid_estimator)])
    contract_prediction = pipeline.predict(test_raw.head(100))
    assert contract_prediction.shape == (100,) and np.isfinite(contract_prediction).all()
    pipeline_path = OUTPUT_DIR/'xgboost_sarima_pipeline.pkl'
    with pipeline_path.open('wb') as file: cloudpickle.dump(pipeline, file)
    run = wandb.init(project=WANDB_PROJECT, name='XGBoost_SARIMA_Best_Model_Registry', job_type='model_registration', reinit=True, config=best)
    artifact = wandb.Artifact('walmart-xgboost-sarima-best-pipeline', type='model', metadata={'model_family':'XGBoost+SARIMA', **best})
    artifact.add_file(str(pipeline_path))
    logged = run.log_artifact(artifact, aliases=['best', 'latest'])
    logged.wait()
    run.link_artifact(logged, target_path=REGISTRY_TARGET, aliases=['champion', 'latest'])
    run.summary.update(best); run.finish(); print(f'Registered: {REGISTRY_TARGET}:champion')